# **Preprocesamiento de audio**
## **Sistema de clasificación de sonidos urbanos para alertas**

**Proyecto:** Detección de sonidos de emergencia en entornos urbanos  
**Equipo:** Alessandra · Natalia · Andrés

### **¿Qué hace este notebook?**

Este notebook tiene dos objetivos:

1. **Ejecutar el preprocesado** de los ≥60.200 audios usando la clase `Preprocess` (`src/data/preprocess.py`) para generar los espectrogramas y features escalares que necesita el modelo.
2. **EDA post-preprocesado**: analizar y visualizar el dataset procesado (`processed_metadata.csv`) para verificar la calidad del proceso antes de entrenar.

### **Salidas que genera el preprocesador**

| Archivo | Descripción |
|---|---|
| `mel_path` | Mel-spectrogram en dB --> `.npy` `[1, 128, T]` |
| `mfcc_path` | Coeficientes MFCC --> `.npy` `[1, 40, T]` |
| `waveform_path` | Waveform normalizada --> `.npy` `[1, samples]` |
| `processed_metadata.csv` | CSV con rutas y features escalares |
| `label_mapping_human_label.pkl` | Mapeo clase --> índice entero |
| `label_mapping_alertable.pkl` | Mapeo alertable --> índice entero |

## **1. Instalación de dependencias**

In [ ]:
pip install librosa torch torchaudio soundfile tqdm matplotlib seaborn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import torch
import torchaudio
import torchaudio.transforms as T
import pickle
import json
import os

from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from typing import Dict, List, Tuple

from src.data.preprocess import Preprocess, PreprocessConfig
from src.data.build.metadata import MetadataEX
import src.data.build.zenodo_ds.zenodo_ds as zenodo
from src.data.build.dataset import Dataset
from src.utils.config import RAW_DIR, INTERIM_DIR

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Estilo de gráficos
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'{torch.cuda.get_device_name(0)}')
print(f'PyTorch {torch.__version__} · torchaudio {torchaudio.__version__}')

## **2. Configuración de rutas**

In [ ]:
PROCESSED_DIR = INTERIM_DIR / 'processed_dataset'
METADATA_CSV = PROCESSED_DIR / 'processed_metadata.csv'
LABEL_MAP_HUMAN = PROCESSED_DIR / 'label_mapping_human_label.pkl'
LABEL_MAP_ALERT = PROCESSED_DIR / 'label_mapping_alertable.pkl'

print('Rutas del proyecto:')
print(f'RAW_DIR --> {RAW_DIR}')
print(f'INTERIM_DIR --> {INTERIM_DIR}')
print(f'PROCESSED --> {PROCESSED_DIR}')
print(f'METADATA CSV --> {METADATA_CSV}')

## **3. Ejecución del preprocesado**

- La clase `Preprocess` realiza las siguientes transformaciones por cada audio:

```
Audio raw  -->  Mono  -->  Resample a 44100 Hz  -->  Pad/Crop a 5s  -->  Normalización peak
                                                                           │
                                          ┌────────────────────────────────┤
                                          ▼                ▼               ▼
                                   mel_db.npy        mfcc.npy       waveform.npy
                                   [1, 128, T]       [1, 40, T]      [1, samples]
```

- **Procesado incremental:** Si los `.npy` ya existen para un audio, el proceso lo omite y pasa al siguiente.

In [ ]:
# CSVs de entrada
CSV_FILES = [
    'UrbanSound8k.csv',
    'audioset.csv',
    'ESC50.csv',
    'zenodo.csv',
    'Guns_DS.csv',
    'VOICe.csv',
    'ravdess_dataset.csv',
]
csv_paths = [RAW_DIR / f for f in CSV_FILES]

# Configuración
config = PreprocessConfig(
    sample_rate=44100,
    n_mels=128,
    n_mfcc=40, # 40 coeficientes para la rama MFCC del modelo dual
    n_fft=2048,
    hop_length=512,
    target_duration=5.0,
    normalize_peak=True,
    augment=False,
    save_waveform=True,
    save_mel=True,
    save_mfcc=True,
)

# Instanciar y ejecutar el preprocesado
pp = Preprocess(
    csv_paths=csv_paths,
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR,
    config=config,
)

print('Configuración activa:')
for k, v in vars(config).items():
    print(f'{k:30s}: {v}')


In [ ]:
# EJECUTAR PREPROCESADO
# Tiempo estimado: ~30 min en CPU.
# Los audios ya procesados se saltan automáticamente.

df_processed = pp.run()

print(f'\nPreprocesado completado.')
print(f'Registros totales en histórico: {len(df_processed):,}')
print(f'Columnas generadas: {list(df_processed.columns)}')
df_processed.head()

---
## **4. EDA post-preprocesado**
- Una vez generado el CSV, analizamos el dataset resultante para detectar posibles problemas antes de entrenar.

### **4.1 Carga del dataset procesado**

In [ ]:
df = pd.read_csv(METADATA_CSV)

print(f'Shape: {df.shape}')
print(f'Filas: {len(df):,}')
print(f'Columnas: {df.shape[1]}')

In [ ]:
print('Tipos de datos:')
print(df.dtypes.to_string())

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# Valores nulos
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)

if nulls.empty:
    print('No hay valores nulos en el dataset procesado.')
else:
    print(f'Columnas con nulos ({len(nulls)}):')
    pct = (nulls / len(df) * 100).round(2)
    display(pd.DataFrame({'nulos': nulls, '%': pct}))

### **4.2 Distribución por dataset de origen**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Conteos por dataset_source
ds_counts = df['dataset_source'].value_counts()
axes[0].bar(ds_counts.index, ds_counts.values,
            color=sns.color_palette('muted', len(ds_counts)))
axes[0].set_title('Muestras por dataset de origen')
axes[0].set_xlabel('Dataset')
axes[0].set_ylabel('Número de muestras')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(ds_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(ds_counts.values, labels=ds_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('muted', len(ds_counts)))
axes[1].set_title('Proporción por dataset')

plt.suptitle('Composición del dataset procesado', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(ds_counts.to_string())

### **4.3 Distribución train / test**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Global
split_counts = df['split'].value_counts()
axes[0].bar(split_counts.index, split_counts.values,
            color=['steelblue', 'salmon'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Train vs. Test (global)')
axes[0].set_ylabel('Muestras')
for i, v in enumerate(split_counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.2f}%)', ha='center', fontsize=10)

# Por dataset
split_by_ds = df.groupby(['dataset_source', 'split']).size().unstack(fill_value=0)
split_by_ds.plot(kind='bar', ax=axes[1], color=['steelblue', 'salmon'], edgecolor='white', linewidth=1)
axes[1].set_title('Train vs. Test por dataset')
axes[1].set_xlabel('')
axes[1].set_ylabel('Muestras')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Split')

plt.tight_layout()
plt.show()

### **4.4 Distribución de clases (`human_label`)**

In [ ]:
class_counts = df['human_label'].value_counts()
n_classes = len(class_counts)
print(f'Número total de clases: {n_classes}')
print(f'Clase más frecuente: {class_counts.index[0]} | ({class_counts.iloc[0]:,} muestras)')
print(f'Clase menos frecuente: {class_counts.index[-1]} | ({class_counts.iloc[-1]:,} muestras)')
print(f'Ratio max/min: {class_counts.iloc[0] / class_counts.iloc[-1]:.2f}x')

In [ ]:
# Top 30 clases más frecuentes
top_n = 30
top_classes = class_counts.head(top_n)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(top_classes)), top_classes.values,
              color=sns.color_palette('viridis', top_n))
ax.set_xticks(range(len(top_classes)))
ax.set_xticklabels(top_classes.index, rotation=45, ha='right', fontsize=8)
ax.set_title(f'Top {top_n} clases más frecuentes')
ax.set_ylabel('Número de muestras')
ax.axhline(class_counts.mean(), color='red', linestyle='--', alpha=0.6, label=f'Media ({class_counts.mean():.0f})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Histograma del desbalanceo
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(class_counts.values, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de frecuencias por clase')
axes[0].set_xlabel('Número de muestras por clase')
axes[0].set_ylabel('Número de clases')
axes[0].axvline(class_counts.mean(), color='red', linestyle='--', label='Media')
axes[0].axvline(class_counts.median(), color='orange', linestyle='--', label='Mediana')
axes[0].legend()

# Curva de Lorenz (desbalanceo acumulado)
sorted_counts = np.sort(class_counts.values)
cum_samples = np.cumsum(sorted_counts) / sorted_counts.sum()
cum_classes = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
axes[1].plot(cum_classes * 100, cum_samples * 100, 'steelblue', linewidth=2)
axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Distribución perfecta')
axes[1].set_title('Curva de Lorenz del desbalanceo')
axes[1].set_xlabel('% de clases acumuladas (de menor a mayor)')
axes[1].set_ylabel('% de muestras acumuladas')
axes[1].legend()
axes[1].fill_between(cum_classes * 100, cum_samples * 100,
                     np.linspace(0, 100, len(sorted_counts)), alpha=0.15, color='steelblue')

plt.tight_layout()
plt.show()

# Clases con menos de 50 muestras
low_count = class_counts[class_counts < 50]
print(f'Clases con menos de 50 muestras: {len(low_count)} / {n_classes}')
if len(low_count) <= 20:
    display(low_count.to_frame('muestras'))

### **4.5 Distribución de clases de alerta**

In [ ]:
alert_cols = [c for c in ['alertable', 'emergency'] if c in df.columns]

if alert_cols:
    fig, axes = plt.subplots(1, len(alert_cols), figsize=(6 * len(alert_cols), 4))
    if len(alert_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, alert_cols):
        counts = df[col].value_counts(dropna=False)
        colors = ['salmon' if str(v) == 'True' else 'steelblue' for v in counts.index]
        ax.bar([str(x) for x in counts.index], counts.values,
               color=colors, edgecolor='white', linewidth=1.5)
        ax.set_title(f'Distribución de `{col}`')
        ax.set_ylabel('Muestras')
        for i, v in enumerate(counts.values):
            ax.text(i, v + 50, f'{v:,}\n({v/len(df)*100:.2f}%)', ha='center', fontsize=10)

    plt.suptitle('Variables de alerta y emergencia', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No se encontraron las columnas `alertable` o `emergency` en el dataset.')

### **4.6 Análisis de la duración de los audios**

In [ ]:
if 'duration_sec' in df.columns:
    dur = df['duration_sec'].dropna()
    print(f'Duración media:   {dur.mean():.2f} s')
    print(f'Duración mediana: {dur.median():.2f} s')
    print(f'Mín / Máx:        {dur.min():.2f} s / {dur.max():.2f} s')
    print(f'Audios > 5 s:     {(dur > 5).sum():,} ({(dur > 5).mean()*100:.2f}%)')
    print(f'Audios < 1 s:     {(dur < 1).sum():,} ({(dur < 1).mean()*100:.2f}%)')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(dur, bins=50, color='steelblue', edgecolor='white')
    axes[0].axvline(5.0, color='red', linestyle='--', label='Target: 5 s')
    axes[0].axvline(dur.mean(), color='orange', linestyle='--', label=f'Media: {dur.mean():.2f} s')
    axes[0].set_title('Distribución de duración (todos los audios)')
    axes[0].set_xlabel('Duración (s)')
    axes[0].set_ylabel('Número de muestras')
    axes[0].legend()

    if 'dataset_source' in df.columns:
        df.boxplot(column='duration_sec', by='dataset_source', ax=axes[1],
                   vert=True, patch_artist=True)
        axes[1].set_title('Duración por dataset')
        axes[1].set_xlabel('')
        axes[1].set_ylabel('Duración (s)')
        axes[1].tick_params(axis='x', rotation=30)
        plt.sca(axes[1])
        plt.title('Duración por dataset')

    plt.suptitle('')
    plt.tight_layout()
    plt.show()
else:
    print('Columna `duration_sec` no encontrada. Omitiendo análisis de duración.')

### **4.7 Features escalares (señal de audio)**

In [ ]:
SCALAR_COLS = [
    'spectral_centroid_mean', 'spectral_centroid_std',
    'spectral_rolloff_mean',  'spectral_rolloff_std',
    'zcr_mean', 'zcr_std',
    'rms_mean', 'rms_std',
    'chroma_mean', 'chroma_std',
]
available_scalars = [c for c in SCALAR_COLS if c in df.columns]
print(f'Features escalares disponibles ({len(available_scalars)}/{len(SCALAR_COLS)}):')
print(available_scalars)

if available_scalars:
    display(df[available_scalars].describe().round(4))

In [ ]:
if available_scalars:
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    axes = axes.flatten()

    palette = sns.color_palette('muted', len(available_scalars))
    for i, col in enumerate(available_scalars):
        axes[i].hist(df[col].dropna(), bins=40, color=palette[i], edgecolor='white')
        axes[i].set_title(col.replace('_', ' '), fontsize=9)
        axes[i].set_xlabel('')
        axes[i].set_yticks([])

    # Ocultar ejes sobrantes
    for j in range(len(available_scalars), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribución de features escalares', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlación entre features escalares
if len(available_scalars) > 1:
    corr = df[available_scalars].corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, ax=ax,
                linewidths=0.5, cbar_kws={'shrink': 0.8})
    ax.set_title('Correlación entre features escalares')
    plt.tight_layout()
    plt.show()